# 38 · 类目、意图与拒识

单元：S05。先修：37。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：人工标签与规则机制；无真实中文用户意图训练。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 标签定义
support需要维修帮助，transaction询价购买，navigation寻找指定入口；不确定时abstain。类目是主题，意图是动作，二者不能互换。

类目以人工device/food主题划分，意图独立；apple可能指实体或食物，必须保留unknown。

In [2]:
examples=[('repair battery','support'),('buy phone','transaction'),('official login','navigation'),('apple','abstain'),('buy battery','abstain'),('weather','abstain')]
print(pd.DataFrame([(q,y,intent(q)) for q,y in examples],columns=['query','expected','predicted']))
for q in ["buy phone","repair phone","buy fruit","apple","phone pie"]:
    print(q,"category",category(q),"intent",intent(q))
assert category("buy phone")==category("repair phone")=="device"
assert intent("buy phone")!=intent("repair phone")
assert category("phone pie")=="unknown"

            query     expected    predicted
0  repair battery      support      support
1       buy phone  transaction  transaction
2  official login   navigation   navigation
3           apple      abstain      abstain
4     buy battery      abstain      abstain
5         weather      abstain      abstain
buy phone category device intent transaction
repair phone category device intent support
buy fruit category food intent transaction
apple category unknown intent abstain
phone pie category unknown intent abstain


## 规则审计与拒识
这是可检查规则，不声称学习出分类边界。覆盖率只以非拒识为分子。预测移除冲突词buy对结果的影响。

In [3]:
assert intent('buy battery')=='abstain' and intent('battery')=='support'
covered=[(q,y) for q,y in examples if intent(q)!='abstain']
print('coverage',len(covered)/len(examples),'covered accuracy',np.mean([intent(q)==y for q,y in covered]))

coverage 0.5 covered accuracy 1.0


## 分布变化
原规则不理解隐含请求与否定。保持规则不变，只换问法，保留失败。

In [4]:
shift=[('how long does it last','support'),('do not buy phone','support')]
for q,y in shift: print(q,'expected',y,'predicted',intent(q))

how long does it last expected support predicted abstain
do not buy phone expected support predicted transaction


## 路由预算
意图只能作为特征或可回退路由，不能武断删光候选。只对明确navigation施加official约束，拒识回通用召回。

In [5]:
for q in ['official repair','apple','official login']:
    label=intent(q); required=['official'] if label=='navigation' else []
    print(q,label,field_rank(q,required=required))

official repair abstain [('a', 3.0), ('d', 3.0), ('c', 2.0), ('b', 1.0)]
apple abstain [('d', 3.0), ('a', 2.0), ('b', 2.0), ('c', 1.0)]
official login navigation [('a', 1.0)]


## 本次保存结果的解读

buy phone与repair phone都属于人工device类目，动作意图不同；phone pie主题冲突返回unknown。buy battery同时触发交易与支持而拒识；否定句do not buy phone仍会误触发交易，暴露词规则缺乏语义。覆盖率/覆盖内准确率只对这几条人工例子成立。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。